# Exemplo – Word Embeddings com fastText (Português)

## Contexto

Esta atividade prática complementa o **Capítulo 3 – Word Embeddings e Representações Distribuídas** da aula de Extração de Características.

O objetivo é explorar, de forma concreta, como palavras podem ser representadas como **vetores em um espaço semântico contínuo**, utilizando **embeddings pré-treinados fastText em língua portuguesa**.  
O foco da atividade não é o treinamento de modelos, mas a **compreensão do comportamento e das propriedades das representações distribuídas**.

Para manter o foco conceitual e reduzir dependências, todas as operações serão realizadas **utilizando apenas NumPy**, sem bibliotecas específicas de PLN.

---

## Objetivos

Ao final desta atividade, o estudante deverá ser capaz de:

- Compreender o papel dos **Word Embeddings** como técnica de extração de características em PLN;
- Utilizar **vetores fastText pré-treinados em português**, compreendendo a relação entre corpus, vocabulário e dimensionalidade;
- Carregar um **subconjunto do vocabulário** de forma controlada, considerando limitações de memória e desempenho;
- Interpretar palavras como **pontos em um espaço vetorial semântico**;
- Calcular **similaridade semântica** por meio do cosseno entre vetores;
- Identificar **vizinhos semânticos** a partir da proximidade vetorial;
- Explorar **analogias vetoriais** do tipo *a : b :: c : ?*, quando aplicável.

---

## Observação didática

Os resultados observados neste exemplo devem ser interpretados como **efeitos estatísticos do corpus de treinamento**, e não como representações linguísticas perfeitas ou semanticamente completas.  
A análise crítica dos resultados faz parte essencial do aprendizado.


## O corpus dos vetores fastText em português (cc.pt.300)

Os vetores utilizados neste exemplo foram extraídos do arquivo:

https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.pt.300.vec.gz

Esses vetores foram treinados pela equipe do fastText (Meta AI) a partir de um grande corpus textual em língua portuguesa, conhecido como Common Crawl (cc).

A nomenclatura `cc.pt.300` do arquivo indica:

- **cc** → *Common Crawl* (origem do corpus);
- **pt** → idioma português;
- **300** → dimensão dos vetores (cada palavra é representada por um vetor de 300 números reais).

### 1) Download do fastText (Português)

O arquivo `cc.pt.300.vec.gz` contém aproximadamente:
- 2 milhões de palavras
- vetores de 300 dimensões
- arquivo compactado ~1,2 GB
- memória necessária para tudo ≈ ~2,4 GB em float32

Ou seja, carregar tudo não é viável em um Colab padrão (nem desejável em aula). Para fins didáticos, carregaremos 200.000 palavras, o que já é suficiente para experimentos em sala.

In [ ]:
import os
import gzip
import numpy as np
from pathlib import Path


In [ ]:
# Fonte oficial de distribuição do fastText (arquivos hospedados em dl.fbaipublicfiles.com)
URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.pt.300.vec.gz"

out_path = Path("cc.pt.300.vec.gz")
if not out_path.exists():
    !wget -q --show-progress -O cc.pt.300.vec.gz $URL
else:
    print("Arquivo já existe:", out_path)


cc.pt.300.vec.gz    100%[===================>]   1.18G  57.6MB/s    in 21s     


### 2) Carregando um subconjunto dos vetores

O arquivo `.vec` possui:
- uma primeira linha com: `<n_palavras> <dimensão>`
- linhas seguintes: `palavra v1 v2 ... v_dim`

Vamos carregar apenas um subconjunto (`max_words`) para economizar memória.

O valor 200.000 foi escolhido por três razões principais:
1)  Compromisso entre cobertura lexical e memória  
    Cada vetor tem:  
      - 300 dimensões
      - float32 → 4 bytes por valor  

    Logo: 300 × 4 bytes ≈ 1,2 KB por palavra  
    Para 200.000 palavras: 200.000 × 1,2 KB ≈ 240 MB  
    Somando overhead de Python (listas, dicionários): ~300–350 MB de RAM

2)  O fastText distribui o arquivo ordenado por frequência no corpus.
    Isso significa que, ao carregar as primeiras 200.000 palavras, obtemos:
    - praticamente todo o vocabulário “útil”:
      - substantivos comuns
      - verbos frequentes
      - adjetivos
      - termos acadêmicos e técnicos gerais
    - excelente cobertura para:
      - exemplos didáticos
      - analogias
      - similaridade semântica
    Na prática:  

    > 200k cobre 95% dos casos relevantes para PLN.

3)  Por que não 100.000 ou 500.000?
    - 100.000: apesar de ser mais leve, pode ter mais OOV (especialmente termos técnicos, nomes próprios, plurais)
    - 500.000: excelente cobertura. Porém, consome de 600-700 MB de RAM e corre o risco de lentidão ou crash.

In [ ]:
def load_fasttext_vec_gz(path, max_words=200_000, dtype=np.float32):
    """
    Carrega um subconjunto do arquivo fastText .vec.gz (Português) em memória.

    Retorna:
      - words: lista de strings
      - W: matriz (n, dim) float32
      - word_to_idx: dicionário palavra->índice
    """
    words = []
    vectors = []

    with gzip.open(path, "rt", encoding="utf-8", newline="\n", errors="ignore") as f:
        header = f.readline().strip().split()
        if len(header) != 2:
            raise ValueError("Header inválido no arquivo .vec")
        total_words, dim = int(header[0]), int(header[1])

        # Lê até max_words (ou até acabar o arquivo)
        for i, line in enumerate(f):
            if i >= max_words:
                break
            parts = line.rstrip().split(" ")
            if len(parts) < dim + 1:
                continue
            word = parts[0]
            vec = np.array(parts[1:dim+1], dtype=dtype)
            words.append(word)
            vectors.append(vec)

    W = np.vstack(vectors)
    word_to_idx = {w: i for i, w in enumerate(words)}
    return words, W, word_to_idx, dim, total_words


In [ ]:
WORDS, W, WORD2IDX, DIM, TOTAL = load_fasttext_vec_gz("cc.pt.300.vec.gz", max_words=200_000)

print("Palavras no arquivo (total):", TOTAL)
print("Palavras carregadas:", len(WORDS))
print("Dimensão:", DIM)
print("Exemplo:", WORDS[0], W[0][:8])


Palavras no arquivo (total): 2000000
Palavras carregadas: 200000
Dimensão: 300
Exemplo: , [ 0.0734  0.0174  0.1548  0.006   0.0116 -0.0678 -0.0254 -0.0116]


### 3) Normalização e funções utilitárias

Nesta seção, fazemos duas coisas fundamentais:

1. **Normalizamos os vetores** (normalização L2) para que a similaridade entre palavras dependa apenas do **ângulo** entre vetores, e não do seu tamanho.
2. Definimos **funções utilitárias** que encapsulam operações comuns: verificação de vocabulário, obtenção de vetor, similaridade por cosseno e consulta dos vizinhos semânticos.

#### 3.1 Por que normalizar?

Embeddings são vetores reais em alta dimensão. Cada vetor possui:

- **direção** (principal componente semântica);
- **magnitude** (norma), que pode variar entre palavras.

Como queremos medir **proximidade semântica**, usamos a **similaridade do cosseno**:

$$
\cos(\theta) = \frac{v_a \cdot v_b}{\|v_a\|\,\|v_b\|}
$$

Ao normalizar cada vetor para ter norma 1 (vetor unitário), obtemos:

$$
\hat{v}_a \cdot \hat{v}_b = \cos(\theta)
$$

Ou seja: **o produto interno passa a ser o próprio cosseno**, e a consulta fica eficiente com operações matriciais (`Wn @ v`).

#### 3.2 O que fazem as funções utilitárias?

- `has_word(word)`: verifica se o termo está no vocabulário carregado (**controle de OOV**).
- `vec(word)`: retorna o vetor **normalizado** do termo.
- `similarity(a, b)`: retorna a similaridade do cosseno entre duas palavras.
- `most_similar(word, topn)`: lista as palavras mais próximas semanticamente no espaço vetorial.

In [ ]:
# Normalização L2: transforma cada vetor em unitário (norma 1)
def normalize_rows(X, eps=1e-12):
    """Normaliza cada linha de X por norma L2 (evita divisão por zero via eps)."""
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, eps)

# Matriz de embeddings normalizados (n_palavras, dim)
Wn = normalize_rows(W)

# Verifica se a palavra está no vocabulário carregado
def has_word(word: str) -> bool:
    return word in WORD2IDX

# Retorna o vetor normalizado associado a uma palavra
def vec(word: str) -> np.ndarray:
    if not has_word(word):
        raise ValueError(f"Palavra fora do vocabulário (OOV): {word}")
    return Wn[WORD2IDX[word]]

# Retorna o cosseno da similaridade entre duas palavras
def similarity(a: str, b: str) -> float:
    """Como os vetores estão normalizados, o produto interno é o cosseno."""
    return float(vec(a) @ vec(b))

# Recupera as palavras mais próximas semanticamente de uma palavra-alvo
def most_similar(word: str, topn: int = 10):
    """Retorna lista [(palavra, similaridade), ...] ordenada por similaridade decrescente."""
    v = vec(word)
    sims = Wn @ v  # similaridade por cosseno com todo o vocabulário

    # Seleciona candidatos mais promissores sem ordenar o vetor inteiro (mais eficiente)
    idx = np.argpartition(-sims, range(topn + 1))[:topn + 1]
    idx = idx[np.argsort(-sims[idx])]

    # Remove o próprio termo da lista final
    out = [(WORDS[i], float(sims[i])) for i in idx if WORDS[i] != word][:topn]
    return out


### 4) Testes rápidos

Observação: dependendo do subconjunto carregado, algumas palavras podem ficar fora do vocabulário (OOV).

In [ ]:
test_terms = ["bom", "professor", "aluno", "computador", "banco", "canote"]

for t in test_terms:
    if has_word(t):
        # lista os 5 mais relevantes
        print(f"\n{t} ->", most_similar(t, topn=5))
    else:
        print(f"\n{t} -> OOV (não carregado no subconjunto)")



bom -> [('ótimo', 0.809251070022583), ('excelente', 0.7200700044631958), ('otimo', 0.7020905017852783), ('péssimo', 0.6887496709823608), ('óptimo', 0.6811461448669434)]

professor -> [('aluno', 0.7751575112342834), ('Professor', 0.7618227601051331), ('educador', 0.7359088659286499), ('ex-professor', 0.7235325574874878), ('professora', 0.6962634325027466)]

aluno -> [('professor', 0.7751575112342834), ('estudante', 0.7093219757080078), ('alunos', 0.6872287392616272), ('Aluno', 0.669069230556488), ('alunado', 0.6674551963806152)]

computador -> [('microcomputador', 0.7279655933380127), ('laptop', 0.7259988188743591), ('Computador', 0.6951974630355835), ('notebook', 0.6891294121742249), ('computadores', 0.6842197775840759)]

banco -> [('bancos', 0.7190471887588501), ('Banco', 0.713119387626648), ('correntista', 0.559371829032898), ('bancário', 0.5278663039207458), ('correntistas', 0.49886131286621094)]

canote -> OOV (não carregado no subconjunto)


### 5) Similaridade entre pares

Do ponto de vista geométrico:

- cada palavra é um ponto em um espaço de 300 dimensões;
- a **distância** ou o **ângulo** entre dois vetores indica similaridade semântica;
- palavras semanticamente próximas tendem a ocupar regiões próximas do espaço.

Assim, quando calculamos a similaridade entre duas palavras, não estamos comparando significados de forma simbólica, mas **medindo relações geométricas** em um espaço vetorial de alta dimensão.


In [ ]:
pairs = [
    ("bom", "ótimo"),
    ("bom", "ruim"),
    ("professor", "aluno"),
    ("computador", "internet"),
    ("dinheiro", "economia"),
    ("avestruz", "bicicleta"),
    ("canote","bicicleta")
]

for a, b in pairs:
    if has_word(a) and has_word(b):
        print(f"similaridade({a},{b}) = {similarity(a,b):.4f}")
    else:
        print(f"({a},{b}) -> OOV em pelo menos um termo")


similaridade(bom,ótimo) = 0.8093
similaridade(bom,ruim) = 0.6663
similaridade(professor,aluno) = 0.7752
similaridade(computador,internet) = 0.4827
similaridade(dinheiro,economia) = 0.3128
similaridade(avestruz,bicicleta) = 0.1526
(canote,bicicleta) -> OOV em pelo menos um termo


### 6) Analogias (a : b :: c : ?)

Uma das propriedades mais conhecidas e conceitualmente interessantes dos *Word Embeddings* é a capacidade de capturar **relações semânticas e sintáticas por meio de operações vetoriais simples**.

Essas relações são frequentemente expressas na forma de **analogias**, do tipo:

> **a : b :: c : ?**  
> (“a está para b assim como c está para ?”)

### Intuição geométrica

Em um espaço vetorial de embeddings bem treinado, certas relações linguísticas tendem a corresponder a **deslocamentos vetoriais aproximadamente constantes**.

Por exemplo, a relação entre os vetores de “rei” e “homem” pode ser semelhante à relação entre “rainha” e “mulher”.

Geometricamente, isso pode ser expresso como:

$$
\vec{rei} - \vec{homem} \approx \vec{rainha} - \vec{mulher}
$$

Reorganizando:

$$
\vec{rainha} \approx \vec{rei} - \vec{homem} + \vec{mulher}
$$

Essa ideia generaliza para a forma:

$$
a : b :: c : ? \quad \Rightarrow \quad \vec{?} \approx \vec{b} - \vec{a} + \vec{c}
$$

### Implementação computacional

No código, a analogia é resolvida em três etapas:

1. **Construção do vetor de analogia**  
   Calcula-se o vetor resultante:
   $$
   \vec{v} = \vec{b} - \vec{a} + \vec{c}
   $$

2. **Normalização do vetor resultante**  
   O vetor $(\vec{v})$ é normalizado para que a comparação com os demais vetores utilize corretamente o cosseno da similaridade.

3. **Busca por vizinhos semânticos**  
   Procura-se, no vocabulário, a palavra cujo vetor seja **mais próximo** de $(\vec{v})$, excluindo explicitamente as palavras `a`, `b` e `c`.

O resultado retornado é uma lista das palavras mais próximas semanticamente do vetor calculado.

### Exemplos clássicos

Alguns exemplos frequentemente utilizados na literatura:

- `homem : rei :: mulher : rainha`
- `brasil : brasileiro :: itália : italiano`
- `bom : melhor :: ruim : pior`

É importante destacar que **nem todas as analogias funcionam perfeitamente**, especialmente em idiomas diferentes do inglês ou quando o corpus de treinamento não contém evidência estatística suficiente para determinada relação.

### Limitações importantes

Apesar de impressionantes, as analogias em embeddings possuem limitações claras:

- Dependem fortemente do **corpus de treinamento**;
- Podem falhar para palavras raras ou fora do vocabulário (OOV);
- Não resolvem ambiguidade semântica (polissemia);
- Funcionam melhor para relações **frequentes e regulares**.

Mesmo assim, esse mecanismo é extremamente valioso do ponto de vista didático, pois evidencia que os embeddings capturam **regularidades estruturais da linguagem** de forma emergente, sem regras explícitas programadas pelo desenvolvedor.


In [ ]:
def analogy(a, b, c, topn=10):
    for w in [a, b, c]:
        if not has_word(w):
            raise ValueError(f"OOV: {w}")

    v = vec(b) - vec(a) + vec(c)
    v = v / max(np.linalg.norm(v), 1e-12)

    sims = Wn @ v
    banned = {a, b, c}

    idx = np.argpartition(-sims, range(topn + 50))[:topn + 50]
    idx = idx[np.argsort(-sims[idx])]

    out = []
    for i in idx:
        w = WORDS[i]
        if w not in banned:
            out.append((w, float(sims[i])))
        if len(out) >= topn:
            break
    return out


In [ ]:
analogy_tests = [
    ("homem", "rei", "mulher"), # homem : rei :: mulher : rainha
    ("brasil", "brasileiro", "itália"), #brasil : brasileiro :: itália : italiano
    ("bom", "melhor", "ruim"), # bom : melhor :: ruim : pior
]

for a, b, c in analogy_tests:
    print(f"\n{a}:{b} :: {c}: ?")
    try:
        print(analogy(a, b, c, topn=8))
    except Exception as e:
        print("Não foi possível:", e)


homem:rei :: mulher: ?
[('rainha', 0.7362514138221741), ('princesa', 0.603180468082428), ('rainha-mãe', 0.5876596570014954), ('monarca', 0.5662566423416138), ('rainhas', 0.552031397819519), ('Rei', 0.5440016388893127), ('Rainha', 0.5381298065185547), ('reis', 0.5158230066299438)]

brasil:brasileiro :: itália: ?
[('italiano', 0.6009321808815002), ('Itália', 0.5478072762489319), ('italiana', 0.5333001613616943), ('milanês', 0.5041080713272095), ('ítalo-brasileiro', 0.4924170970916748), ('europeu', 0.4689832329750061), ('italianos', 0.4619034230709076), ('francês', 0.45827603340148926)]

bom:melhor :: ruim: ?
[('pior', 0.7218661308288574), ('péssima', 0.553585410118103), ('horrível', 0.5532447099685669), ('piores', 0.5184032917022705), ('diferente', 0.49898460507392883), ('horrorosa', 0.4978390038013458), ('melho', 0.49676087498664856), ('piorar', 0.492562860250473)]
